# qPISA EV6: substrate–product time courses with MEROPS-position windows

Read the four defined GLP-1 substrate/product pairs and retain their measured signals and cut contexts. Associate each DPP4 observation with its exact time- and replicate-matched buffer control.

## Definitions

Input: `44320_2024_71_MOESM7_ESM.xlsx`, sheet `Product peptide intensities`, DOI `10.1038/s44320-024-00071-4`. Adjacent original columns define each substrate/product pair. Parent sequences differ from their products by the two N-terminal residues; window positions are `--AB|CDEF` and `site_evidence=source_substrate_product_pair`.

`source_pairs` retains both raw intensities at each source condition; `source_measurements` exposes the individual intensity records with unit `instrument_signal_au`. `measurements` contains control-matched substrate log2 depletion and product log2 enrichment. Values are calculated from signal ratios at the same assay pair, time and replicate. These are quantitative measurements, without fitting a kinetic model or applying a cleavage threshold.

The deposit has buffer controls at 0, 30, 60 and 120 min, and DPP4 observations at 30, 60 and 120 min, each with replicates A–C. No missing DPP4 zero-time row is invented. Four source assay pairs remain separate even when their product sequence is identical. No CSV or intermediate files are written.

In [1]:
from pathlib import Path
import sys
import logging

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "assets/tools/proteolysis_data_manager").is_dir())
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
RAW = ROOT / "data/proteolysis/window_neighbourhood_datasets/raw"
pd.set_option("display.max_columns", None)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.qpisa").setLevel(logging.WARNING)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.pics").setLevel(logging.WARNING)

from proteolysis_data_manager.analysis.window_neighbourhood.qpisa import read_timecourse_measurements
q = read_timecourse_measurements(RAW / "dpp4_qpisa")
source_pairs = q["pairs"]
changes = q["changes"]

# Expose all source intensities, preserving the original assay-pair identities.
signal_columns = ["substrate_signal_au", "product_signal_au"]
source_measurements = source_pairs.melt(
    id_vars=[column for column in source_pairs if column not in signal_columns],
    value_vars=signal_columns, var_name="measurement_type", value_name="value",
)
source_measurements["unit"] = "instrument_signal_au"
source_measurements["measured_sequence"] = np.where(
    source_measurements["measurement_type"].eq("substrate_signal_au"),
    source_measurements["substrate_sequence"], source_measurements["product_sequence"],
)

# Normalize the matched quantitative changes to the same measurement/window view.
change_columns = ["substrate_log2_depletion", "product_log2_enrichment"]
measurements = changes.melt(
    id_vars=[column for column in changes if column not in change_columns],
    value_vars=change_columns, var_name="measurement_type", value_name="value",
)
measurements["unit"] = "log2_fold_change"
measurements["duration_s"] = measurements["time_s"]
measurements["measured_sequence"] = np.where(
    measurements["measurement_type"].eq("substrate_log2_depletion"),
    measurements["substrate_sequence"], measurements["product_sequence"],
)
measurements["source_record_id"] = ("EV6/" + measurements["assay_pair"].astype(str) + "/"
    + measurements["time_s"].astype(str) + "/" + measurements["replicate"] + "/" + measurements["measurement_type"])

In [2]:
overview = pd.DataFrame([
    ("Defined substrate/product pairs", source_pairs["assay_pair"].nunique()),
    ("Source pair-condition rows", len(source_pairs)),
    ("Source intensity observations", len(source_measurements)),
    ("DPP4 pair/time/replicate observations with a matched control", len(changes)),
    ("Signed quantitative changes (substrate and product)", len(measurements)),
    ("Distinct MEROPS-position windows", source_pairs["window"].nunique()),
], columns=["evidence_unit", "count"])
display(overview)
display(source_pairs[["assay_pair", "substrate_sequence", "product_sequence", "window", "site_evidence"]].drop_duplicates().reset_index(drop=True))
assert measurements["source_record_id"].is_unique and np.isfinite(measurements["value"]).all()
assert source_pairs["substrate_sequence"].str[2:].eq(source_pairs["product_sequence"]).all()
assert changes["control_source_excel_row"].notna().all()
assert not measurements["time_s"].eq(0).any()
assert np.allclose(changes["substrate_log2_depletion"], -np.log2(changes["substrate_signal_ratio"]), atol=1e-12, rtol=0)

,evidence_unit,count
0,Defined substrate/product pairs,4
1,Source pair-condition rows,84
2,Source intensity observations,168
3,DPP4 pair/time/replicate observations with a m...,36
4,Signed quantitative changes (substrate and pro...,72
5,Distinct MEROPS-position windows,4


,assay_pair,substrate_sequence,product_sequence,window,site_evidence
0,1,HAEGTFTSDVSR,EGTFTSDVSR,--HA|EGTF,source_substrate_product_pair
1,2,DAEGTFTSDVSR,EGTFTSDVSR,--DA|EGTF,source_substrate_product_pair
2,3,HAPGTFTSDVSR,PGTFTSDVSR,--HA|PGTF,source_substrate_product_pair
3,4,DAPGTFTSDVSR,PGTFTSDVSR,--DA|PGTF,source_substrate_product_pair


## Inspect quantitative values for a defined window

Select an assay pair below. All fields and both source intensities stay in `measurements`; the preview shows the value, time, replicate, window, and enzyme/control source rows. `relative_substrate_signal_loss` also provides the fractional decrease in signal, including negative values. Signal loss is not relabelled as a calibrated amount of peptide.

In [3]:
ASSAY_PAIR = 1
PREVIEW_ROWS = 9

selected_measurements = measurements.loc[measurements["assay_pair"].eq(ASSAY_PAIR)].copy()
columns = ["enzyme", "window", "time_s", "replicate", "measurement_type", "value", "unit", "source_excel_row", "control_source_excel_row"]
display(selected_measurements.groupby("measurement_type", sort=False).head(PREVIEW_ROWS)[columns].reset_index(drop=True))

,enzyme,window,time_s,replicate,measurement_type,value,unit,source_excel_row,control_source_excel_row
0,DPP4,--HA|EGTF,1800,A,substrate_log2_depletion,8.101347,log2_fold_change,15,6
1,DPP4,--HA|EGTF,1800,B,substrate_log2_depletion,7.830519,log2_fold_change,16,7
2,DPP4,--HA|EGTF,1800,C,substrate_log2_depletion,7.623925,log2_fold_change,17,8
3,DPP4,--HA|EGTF,3600,A,substrate_log2_depletion,8.080176,log2_fold_change,18,9
4,DPP4,--HA|EGTF,3600,B,substrate_log2_depletion,8.701698,log2_fold_change,19,10
5,DPP4,--HA|EGTF,3600,C,substrate_log2_depletion,8.857475,log2_fold_change,20,11
6,DPP4,--HA|EGTF,7200,A,substrate_log2_depletion,7.729116,log2_fold_change,21,12
7,DPP4,--HA|EGTF,7200,B,substrate_log2_depletion,8.339219,log2_fold_change,22,13
8,DPP4,--HA|EGTF,7200,C,substrate_log2_depletion,8.814138,log2_fold_change,23,14
9,DPP4,--HA|EGTF,1800,A,product_log2_enrichment,9.861782,log2_fold_change,15,6


### Remarks

Retain source intensities alongside relative changes. Product signals belonging to different designed substrates must not be pooled by product sequence alone. This table records measured dynamics with an explicit source pair, not an assumed kinetic constant.

### Notes

## Results / Summary

### LLM

The output links all four defined source cut contexts to original time-series signals and their control-matched signed changes. The same window coordinates and measurement-type distinction can be used alongside PICS QF and the larger qPISA endpoint tables.

### Person

## Common numerical parameters

The common columns are remaining percent = 100 × 2⁻ᵈ and depletion percent = 100 × (1 − 2⁻ᵈ), where d is the reported log₂ control/treated difference. Negative net loss remains negative. The apparent exponential rate is ln(2) × d/time. These describe control-relative signals. Endpoint candidate-window frequencies use positive fractional depletion as a continuous weight under the approximately mixed input assumption; they are not thresholded cleavage counts. All transformations stay in memory; no CSV or database is written.

In [4]:
# Use the independent data library from this checkout.
import sys
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.integration.merge import (
    build_dataset, parse_dataset, add_merops_scores,
)
contract = parse_dataset("qpisa_ev6", ROOT / "data/proteolysis")
contract = add_merops_scores(
    contract, ROOT / "data/proteolysis/window_neighbourhood_datasets/raw/merops",
    dataset_index=34,
)
common_parameters = contract.to_frame()
preview = common_parameters.loc[common_parameters.substrate_depletion_percent.notna()]
display(preview.groupby(["source_dataset", "enzyme_name"], dropna=False, sort=False).head(2)[[
    "source_db", "source_dataset", "enzyme_name", "window", "time_s",
    "substrate_remaining_percent", "substrate_depletion_percent", "apparent_rate_s_inv",
    "position_probability_sum", "probability_positions_scored",
]])


,source_db,source_dataset,enzyme_name,window,time_s,substrate_remaining_percent,substrate_depletion_percent,apparent_rate_s_inv,position_probability_sum,probability_positions_scored
24,qpisa,EV6,DPP4,- - H A | E G T F,1800,0.364126,99.635874,0.003120,0.76978,6
26,qpisa,EV6,DPP4,- - H A | E G T F,1800,0.439318,99.560682,0.003015,0.76978,6
